# 🤖 OS Agent — APK Derleme (Buildozer + Google Colab)

Bu notebook, projeyi baştan sona derleyip **kurulabilir bir Android APK'sı** üretir.
- Termux kurulumu **gerekmez** — çalışma zamanında da gerekmez
- Android SDK/NDK, Buildozer tarafından otomatik indirilir (bu notebook'ta manuel SDK kurulumu YOKTUR — bu kasıtlıdır, buildozer'ın kendi indirdiği sürümler en uyumlu olanlardır)
- Yerel AI modeli APK'ya gömülmez; uygulama ilk açılışta kendisi indirir
- llama-cpp-python **varsayılan olarak derlenmez** (isteğe bağlı, bkz. Adım 6)

**Tahmini süre:** ilk derleme 25–45 dakika (SDK/NDK indirme + Gradle ilk kurulum dahil). Sonraki derlemeler çok daha hızlıdır.

> ⚠️ Colab oturumu kapanırsa (`.buildozer/` klasörü kaybolur) SDK/NDK yeniden inecektir. Uzun bir build için Colab sekmesini açık tutun.

## ⚙️ Adım 1 — Sistem Bağımlılıklarını Kur

In [ ]:
%%bash
set -e
sudo apt-get update -qq
sudo apt-get install -y -qq \
    git zip unzip openjdk-17-jdk wget curl \
    python3-pip autoconf automake libtool pkg-config m4 \
    zlib1g-dev libncurses5-dev libncursesw5-dev \
    libssl-dev libreadline-dev libffi-dev libsqlite3-dev \
    cmake ninja-build build-essential libltdl-dev lld
sudo update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java || true
java -version
echo '✅ Sistem bağımlılıkları hazır'

## 📦 Adım 2 — Buildozer ve python-for-android Kur

Sürümler bilinçli olarak pinlendi: Cython 3.x, birçok mevcut p4a
recipe'iyle (özellikle Kivy'nin kendi Cython dosyalarıyla) uyumsuzdur
ve Colab'daki en yaygın "BUILD FAILED" nedenlerinden biridir.

In [ ]:
%%bash
set -e
pip install -q --upgrade pip
pip install -q "Cython==0.29.36" virtualenv
pip install -q "buildozer==1.5.0" "python-for-android==2024.1.21"
buildozer --version
python3 -c "import Cython; print('Cython', Cython.__version__)"
echo '✅ Buildozer hazır'

## 📁 Adım 3 — Proje Dosyalarını Yükle

**Seçenek A:** GitHub reposundan klonla (aşağıdaki hücreyi düzenleyip çalıştırın)

```bash
git clone https://github.com/KULLANICI_ADI/os-agent.git /content/os-agent
```

**Seçenek B (varsayılan):** Bilgisayarınızdan proje zip'ini yükleyin ↓ (aşağıdaki hücreyi çalıştırın, açılan pencereden `.zip` dosyasını seçin)

In [ ]:
# Seçenek B — Zip yükleme (zip içinde hangi isimde klasör olursa olsun
# /content/os-agent altına doğru şekilde taşınır)
from google.colab import files
import zipfile, os, shutil

HEDEF = '/content/os-agent'

print('Proje .zip dosyasını seçin...')
uploaded = files.upload()
zip_adi = list(uploaded.keys())[0]

gecici = '/content/_osagent_extract'
if os.path.isdir(gecici):
    shutil.rmtree(gecici)
os.makedirs(gecici, exist_ok=True)

with zipfile.ZipFile(zip_adi, 'r') as z:
    z.extractall(gecici)

# zip'in içeriği doğrudan dosyalar mı yoksa tek bir alt klasör mü kontrol et
icerik = [p for p in os.listdir(gecici) if not p.startswith('__MACOSX')]
if len(icerik) == 1 and os.path.isdir(os.path.join(gecici, icerik[0])):
    kaynak = os.path.join(gecici, icerik[0])
else:
    kaynak = gecici

if os.path.isdir(HEDEF):
    shutil.rmtree(HEDEF)
shutil.move(kaynak, HEDEF)

assert os.path.isfile(os.path.join(HEDEF, 'buildozer.spec')), (
    'buildozer.spec bulunamadı — zip içeriğini kontrol edin: ' + str(os.listdir(HEDEF))
)
print('✅ Proje şuraya yüklendi:', HEDEF)
print('İçerik:', os.listdir(HEDEF))

## 🧩 Adım 4 — Görsel Varlıkları Doğrula

Proje ile birlikte gerçek `assets/icon.png` ve `assets/splash.png`
dosyaları gelir. Bu hücre yalnızca bunlar eksikse (ör. elle bir zip
hazırladıysanız) yedek bir görsel üretir — normal koşullarda hiçbir
şey yapmaz.

In [ ]:
import os
ASSETS = '/content/os-agent/assets'
os.makedirs(ASSETS, exist_ok=True)

icon_var = os.path.join(ASSETS, 'icon.png')
splash_var = os.path.join(ASSETS, 'splash.png')

if os.path.isfile(icon_var) and os.path.isfile(splash_var):
    print('✅ icon.png ve splash.png zaten mevcut, yeni görsel oluşturulmadı.')
else:
    try:
        from PIL import Image, ImageDraw
        if not os.path.isfile(icon_var):
            img = Image.new('RGB', (512, 512), (15, 15, 20))
            ImageDraw.Draw(img).ellipse([56, 56, 456, 456], fill=(97, 120, 255))
            img.save(icon_var)
            print('İkon oluşturuldu (yedek).')
        if not os.path.isfile(splash_var):
            splash = Image.new('RGB', (720, 1280), (15, 15, 20))
            ImageDraw.Draw(splash).ellipse([260, 460, 460, 660], fill=(97, 120, 255))
            splash.save(splash_var)
            print('Splash oluşturuldu (yedek).')
    except Exception as e:
        print(f'Uyarı: görsel oluşturulamadı ({e}) — buildozer.spec dosya adlarının doğru olduğundan emin olun.')

## 🏗️ Adım 5 — APK Derle (varsayılan: model desteği olmadan, STUB motoruyla)

Bu hücre gerçek derlemeyi başlatır. `android.accept_sdk_license = True`
buildozer.spec içinde tanımlı olduğundan lisans onayı otomatiktir; yine
de olası ek etkileşimli komut istemlerine karşı `yes |` ile besleniyor.

**İlk çalıştırmada** Buildozer; Android SDK cmdline-tools, platform araçları
ve NDK r25b'yi kendi `.buildozer/` önbelleğine indirir (birkaç GB, biraz
zaman alır) — bu normaldir.

In [ ]:
%%bash
set -o pipefail
cd /content/os-agent
yes | buildozer -v android debug 2>&1 | tee /content/buildozer_log.txt
echo ''
echo '=== Üretilen APK Dosyaları ==='
ls -lh bin/*.apk 2>/dev/null || echo '❌ APK bulunamadı — /content/buildozer_log.txt dosyasının SON 100 satırını inceleyin (bir sonraki hücre otomatik gösterir).'

In [ ]:
# Derleme başarısız olduysa bu hücre logun son kısmını gösterir — hatayı
# hızlıca teşhis etmek için genelde yeterlidir.
import os, glob
apks = glob.glob('/content/os-agent/bin/*.apk')
if apks:
    print('✅ Derleme başarılı, APK bulundu:', apks)
else:
    print('❌ APK bulunamadı. Son 100 log satırı:\n')
    if os.path.isfile('/content/buildozer_log.txt'):
        with open('/content/buildozer_log.txt') as f:
            satirlar = f.readlines()
        print(''.join(satirlar[-100:]))
    else:
        print('Log dosyası da bulunamadı — Adım 5 çalıştırıldı mı?')

## 🧪 Adım 6 (İSTEĞE BAĞLI) — Yerel AI Modeli (llama-cpp-python) ile Derleme

**Varsayılan olarak atlayın.** Adım 5'te üretilen APK, model olmadan da
kural-tabanlı motorla (wifi/ses/parlaklık/konum/pil otomasyonları vb.)
tam işlevseldir; kullanıcı uygulama içinden istediği zaman bir GGUF model
indirip gerçek NLU motorunu devreye alabilir — bunun için `llama-cpp-python`
APK'ya gömülü olmak ZORUNDA değildir.

Yine de llama-cpp-python'ı APK'nın kendisine derlemek isterseniz (deneysel,
başarısız olma ihtimali yüksektir — bkz. `p4a-recipes/README.md`):

1. Aşağıdaki hücreyi çalıştırarak `requirements` satırına ekleyin.
2. Temiz bir derleme başlatın (önbellek çakışmalarını önlemek için).

In [ ]:
# Bu hücreyi SADECE deneysel model-içerir derleme istiyorsanız çalıştırın.
spec_yolu = '/content/os-agent/buildozer.spec'
with open(spec_yolu) as f:
    icerik = f.read()

if 'llama-cpp-python' not in icerik:
    icerik = icerik.replace(
        'requirements = python3==3.11,kivy==2.3.0,kivymd==1.2.0,pillow,android,pyjnius,plyer,certifi',
        'requirements = python3==3.11,kivy==2.3.0,kivymd==1.2.0,pillow,android,pyjnius,plyer,certifi,llama-cpp-python',
    )
    with open(spec_yolu, 'w') as f:
        f.write(icerik)
    print('✅ llama-cpp-python requirements listesine eklendi.')
else:
    print('llama-cpp-python zaten listede.')
print('Şimdi: !cd /content/os-agent && buildozer android clean && yes | buildozer -v android debug')

## 📥 Adım 7 — APK'yı Bilgisayara İndir

In [ ]:
import glob, os
from google.colab import files

apk_dosyalari = glob.glob('/content/os-agent/bin/*.apk')
if not apk_dosyalari:
    print('❌ APK bulunamadı. Yukarıdaki Adım 5/5b log çıktısını ve hata tablosunu inceleyin.')
else:
    for apk in apk_dosyalari:
        boyut_mb = os.path.getsize(apk) / 1_048_576
        print(f'📦 {os.path.basename(apk)}  ({boyut_mb:.1f} MB)')
        files.download(apk)
    print('✅ İndirme başladı!')

## 📱 Adım 8 — Telefona Kur ve Kullan

1. APK'yı telefona aktarın (USB, WhatsApp, Google Drive…)
2. **Ayarlar → Güvenlik → Bilinmeyen Kaynaklar**'ı açın
3. APK dosyasına dokunun ve kurun
4. Uygulamayı açın — **model seçim ekranı** karşılar
5. Bir model seçin (internet bağlantısı gerekir, yalnızca ilk seferinde) veya modelsiz devam edin — otomasyonlar STUB motoruyla çalışmaya devam eder
6. Model indikten sonra tamamen çevrimdışı çalışır!

---

## 🛠️ Sık Karşılaşılan Hatalar

| Hata | Çözüm |
|------|-------|
| `SDK license not accepted` | `buildozer.spec` içinde `android.accept_sdk_license = True` zaten var; yine de olursa hücreyi tekrar çalıştırın (`yes \|` ilk seferde tüm promptları henüz göremeyebilir) |
| `NDK version mismatch` / `NDK not found` | `buildozer.spec` → `android.ndk = 25b` satırını değiştirmeyin; `buildozer android clean` sonrası tekrar deneyin |
| `Cython` derleme hatası | Adım 2'deki `Cython==0.29.36` pininin gerçekten kurulduğunu doğrulayın (`pip show cython`) |
| `BUILD FAILED` (Gradle) | Log'da `FAILURE:` ile başlayan satırı arayın — genelde eksik bir `android.permissions` girdisi veya AndroidX bağımlılığıdır |
| APK kurulmuyor | Telefonda **Bilinmeyen kaynak** izni kapalı olabilir; ya da eski bir `osagent` sürümü zaten kurulu olabilir (kaldırıp tekrar deneyin — imza farkı kurulumu engeller) |
| Model indirme başlamıyor / SSL hatası | `requirements`'ta `certifi` bulunduğundan emin olun (bu proje varsayılan olarak ekliyor); telefonun internet bağlantısını kontrol edin |
| Servis bildirimi görünmüyor / arka plan durmuş | Android, pil optimizasyonu ile arka plan servisini durdurmuş olabilir — telefon Ayarlar → Pil → OS Agent → "Kısıtlama Yok" yapın |
| `llama-cpp-python` derlemesi başarısız (Adım 6) | Beklenen bir durumdur — Adım 6'yı atlayıp modelsiz derlemeye dönün, uygulama STUB motoruyla çalışmaya devam eder |